# Baseline reproduction: BERT + GRU (3-class)

Reproduces `Codes/3-class/BERT_w_GRU.ipynb` from the base paper (Islam et al., ICCIT 2020) without the deprecated `torchtext.legacy` API, so it runs on current Kaggle images.

**Before running:**
1. Enable GPU: Notebook Settings -> Accelerator -> GPU T4 x2 (or P100).
2. Add Data -> upload `train.csv` and `test.csv` from `Dataset/` as a private Kaggle dataset (or drag-drop them as Notebook input). These are already the fully preprocessed 3-class splits (14,852 train / 3,001 test rows, columns `Data`,`Sentiment` with labels 0/1/2) — no need to re-run `PreProcess.ipynb`.
3. Update `DATA_DIR` below to match wherever Kaggle mounts your uploaded files (check the Input panel on the right — usually `/kaggle/input/<dataset-slug>/`).

**Target:** base paper reports ~60% test accuracy for 3-class BERT+GRU. This notebook prints a pass/fail against that threshold at the end.


In [ ]:
!pip install -q transformers==4.42.4

In [ ]:
import os, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
# ---- point this at your uploaded dataset ----
DATA_DIR = '/kaggle/input/bengali-sentiment-3class'   # <-- EDIT to your actual Kaggle input folder
TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')
TEST_CSV  = os.path.join(DATA_DIR, 'test.csv')

MAX_LEN     = 400
BATCH_SIZE  = 32
HIDDEN_DIM  = 350
N_LAYERS    = 1
BIDIRECTIONAL = True
DROPOUT     = 0.5
OUTPUT_DIM  = 3
N_EPOCHS    = 20
LR          = 5e-4
WEIGHT_DECAY = 1e-6
BASELINE_ACC = 0.60

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

pad_token_idx = tokenizer.pad_token_id
cls_token_idx = tokenizer.cls_token_id
sep_token_idx = tokenizer.sep_token_id

def encode(text):
    # same truncation the original notebook used: room for [CLS] and [SEP]
    tokens = tokenizer.tokenize(str(text))[:MAX_LEN - 2]
    ids = [cls_token_idx] + tokenizer.convert_tokens_to_ids(tokens) + [sep_token_idx]
    return ids

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

train_df.columns = ['Data', 'Sentiment']
test_df.columns  = ['Data', 'Sentiment']
train_df = train_df.dropna(subset=['Data', 'Sentiment']).reset_index(drop=True)
test_df  = test_df.dropna(subset=['Data', 'Sentiment']).reset_index(drop=True)
train_df['Sentiment'] = train_df['Sentiment'].astype(int)
test_df['Sentiment']  = test_df['Sentiment'].astype(int)

# 85/15 train/val split, same ratio and seed as the original notebook
train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
val_cut = int(len(train_df) * 0.85)
val_df   = train_df.iloc[val_cut:].reset_index(drop=True)
train_df = train_df.iloc[:val_cut].reset_index(drop=True)

print(f'train: {len(train_df)}  val: {len(val_df)}  test: {len(test_df)}')
print('label distribution (train):', train_df.Sentiment.value_counts().to_dict())

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, df):
        self.texts = df['Data'].tolist()
        self.labels = df['Sentiment'].tolist()
        self.encoded = [encode(t) for t in self.texts]
        self.lengths = [len(e) for e in self.encoded]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.encoded[idx], self.labels[idx]

def make_collate(pad_idx):
    def collate(batch):
        seqs, labels = zip(*batch)
        max_len = max(len(s) for s in seqs)
        padded = torch.full((len(seqs), max_len), pad_idx, dtype=torch.long)
        for i, s in enumerate(seqs):
            padded[i, :len(s)] = torch.tensor(s, dtype=torch.long)
        return padded, torch.tensor(labels, dtype=torch.long)
    return collate

collate_fn = make_collate(pad_token_idx)

train_ds = SentimentDataset(train_df)
val_ds   = SentimentDataset(val_df)
test_ds  = SentimentDataset(test_df)

class BucketBatchSampler(torch.utils.data.Sampler):
    """Approximates torchtext's BucketIterator: sort by length to minimize
    padding, then shuffle the resulting batches each epoch."""
    def __init__(self, lengths, batch_size, shuffle=True):
        self.lengths = lengths
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __iter__(self):
        idx = sorted(range(len(self.lengths)), key=lambda i: self.lengths[i])
        batches = [idx[i:i+self.batch_size] for i in range(0, len(idx), self.batch_size)]
        if self.shuffle:
            random.shuffle(batches)
        return iter(batches)

    def __len__(self):
        return (len(self.lengths) + self.batch_size - 1) // self.batch_size

train_loader = DataLoader(train_ds, batch_sampler=BucketBatchSampler(train_ds.lengths, BATCH_SIZE, shuffle=True), collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_sampler=BucketBatchSampler(val_ds.lengths, BATCH_SIZE, shuffle=False), collate_fn=collate_fn)
test_loader  = DataLoader(test_ds, batch_sampler=BucketBatchSampler(test_ds.lengths, BATCH_SIZE, shuffle=False), collate_fn=collate_fn)

In [ ]:
bert = BertModel.from_pretrained('bert-base-multilingual-cased')

class BERTGRUSentiment(nn.Module):
    def __init__(self, bert, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super().__init__()
        self.bert = bert
        embedding_dim = bert.config.to_dict()['hidden_size']
        self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers=n_layers,
                           bidirectional=bidirectional, batch_first=True,
                           dropout=0 if n_layers < 2 else dropout)
        self.out = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        with torch.no_grad():
            embedded = self.bert(text)[0]
        _, hidden = self.rnn(embedded)
        if self.rnn.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1))
        else:
            hidden = self.dropout(hidden[-1, :, :])
        return self.out(hidden)

model = BERTGRUSentiment(bert, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, BIDIRECTIONAL, DROPOUT)

# freeze BERT, only the GRU head trains -- matches the base paper's approach
for name, param in model.named_parameters():
    if name.startswith('bert'):
        param.requires_grad = False

def count_parameters(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f'trainable parameters: {count_parameters(model):,}')

model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss().to(device)

In [ ]:
def accuracy(preds, y):
    rounded_preds = torch.max(preds, 1)[1]
    correct = (rounded_preds == y).float()
    return correct.sum() / len(correct)

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    epoch_loss, epoch_acc = 0.0, 0.0
    all_preds, all_targets = [], []
    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for text, labels in loader:
            text, labels = text.to(device), labels.to(device)
            if is_train:
                optimizer.zero_grad()
            preds = model(text)
            loss = criterion(preds, labels)
            acc = accuracy(preds, labels)
            if is_train:
                loss.backward()
                optimizer.step()
            epoch_loss += loss.item()
            epoch_acc += acc.item()
            all_preds.extend(torch.max(preds, 1)[1].cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
    return epoch_loss / len(loader), epoch_acc / len(loader), all_preds, all_targets

def epoch_time(start, end):
    elapsed = end - start
    return int(elapsed / 60), int(elapsed - int(elapsed / 60) * 60)

In [ ]:
best_valid_loss = float('inf')
CKPT_PATH = '/kaggle/working/bert_gru_3class_best.pt'

for epoch in range(N_EPOCHS):
    start = time.time()
    train_loss, train_acc, _, _ = run_epoch(model, train_loader, criterion, optimizer)
    valid_loss, valid_acc, _, _ = run_epoch(model, val_loader, criterion)
    end = time.time()

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), CKPT_PATH)

    mins, secs = epoch_time(start, end)
    print(f'Epoch {epoch+1:02}/{N_EPOCHS} | {mins}m {secs}s | '
          f'Train Loss: {train_loss:.3f} Acc: {train_acc*100:.2f}% | '
          f'Val Loss: {valid_loss:.3f} Acc: {valid_acc*100:.2f}%')

Load the checkpoint with the best validation loss and evaluate on the held-out test set.

In [ ]:
model.load_state_dict(torch.load(CKPT_PATH))
test_loss, test_acc, test_preds, test_targets = run_epoch(model, test_loader, criterion)

print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')
print()
if test_acc >= BASELINE_ACC:
    print(f'PASS: {test_acc*100:.2f}% >= {BASELINE_ACC*100:.0f}% baseline target')
else:
    print(f'FAIL: {test_acc*100:.2f}% < {BASELINE_ACC*100:.0f}% baseline target')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(test_targets, test_preds, digits=4))
print(confusion_matrix(test_targets, test_preds))